In [1]:
# !pip install -q tensorflow
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist

In [5]:
train_images_count = 1000
test_images_count = 10000
pixels_per_image = 28 * 28
digits_num = 10

(x_train, y_train), (x_test, y_test) = mnist.load_data()

train_images = x_train[0:train_images_count].reshape(train_images_count, pixels_per_image) / 255.0
train_labels_raw = y_train[0:train_images_count]

test_images = x_test[0:test_images_count].reshape(test_images_count, pixels_per_image) / 255.0
test_labels_raw = y_test[0:test_images_count]

def to_one_hot(labels, classes=digits_num):
    one_hot = np.zeros((len(labels), classes), dtype=np.float32)
    for i, label in enumerate(labels):
        one_hot[i][label] = 1
    return one_hot

train_labels = to_one_hot(train_labels_raw)
test_labels = to_one_hot(test_labels_raw)

print("train_images:", train_images.shape)
print("test_images: ", test_images.shape)
print("train_labels:", train_labels.shape)

train_images: (1000, 784)
test_images:  (10000, 784)
train_labels: (1000, 10)


In [7]:
def relu(x):
    return (x < 0) * x

def reluderiv(x):
    return (x > 0)

def train_and_test(hidden_size, learning_rate=0.01, num_epoch=100, seed=42, verbose_every=10):
    np.random.seed(seed)

    weight_hid = 0.2 * np.random.random((pixels_per_image, hidden_size)) - 0.1
    weight_out = 0.2 * np.random.random((hidden_size, digits_num)) - 0.1

    for epoch in range(num_epoch):
        correct_answers = 0

        for j in range(len(train_images)):
            layer_in = train_images[j:j + 1]
            layer_hid = relu(np.dot(layer_in, weight_hid))
            layer_out = np.dot(layer_hid, weight_out)

            correct_answers += int(np.argmax(layer_out) == np.argmax(train_labels[j:j + 1]))

            layer_out_delta = layer_out - train_labels[j:j + 1]
            layer_hid_delta = layer_out_delta @ weight_out.T * reluderiv(layer_hid)

            weight_out -= learning_rate * layer_hid.T.dot(layer_out_delta)
            weight_hid -= learning_rate * layer_in.T.dot(layer_hid_delta)

        train_acc = correct_answers * 100 / len(train_images)

        if verbose_every and ((epoch + 1) % verbose_every == 0 or epoch == num_epoch - 1):
            print(f"  epoch {epoch + 1:3d}/{num_epoch} | train_acc={train_acc:6.2f}%")

    correct_answers = 0
    for j in range(len(test_images)):
        layer_in = test_images[j:j + 1]
        layer_hid = relu(np.dot(layer_in, weight_hid))
        layer_out = np.dot(layer_hid, weight_out)
        correct_answers += int(np.argmax(layer_out) == np.argmax(test_labels[j:j + 1]))

    test_acc = correct_answers * 100 / len(test_images)
    return train_acc, test_acc

In [8]:
hidden_sizes = [10, 50, 100, 200]
results = []

for hidden_size in hidden_sizes:
    print()
    print(f"hidden_size = {hidden_size}")
    train_acc, test_acc = train_and_test(hidden_size, num_epoch=100)
    results.append((hidden_size, train_acc, test_acc))
    print(f"train={train_acc:.2f}%  test={test_acc:.2f}%")

print()
print("Итог:")
print(f"{'hidden_size':>12} | {'train_acc':>10} | {'test_acc':>10}")
for hs, tr, te in results:
    print(f"{hs:>12} | {tr:>9.2f}% | {te:>9.2f}%")


hidden_size = 10
  epoch  10/100 | train_acc= 36.30%
  epoch  20/100 | train_acc= 36.60%
  epoch  30/100 | train_acc= 36.90%
  epoch  40/100 | train_acc= 36.80%
  epoch  50/100 | train_acc= 36.90%
  epoch  60/100 | train_acc= 36.90%
  epoch  70/100 | train_acc= 36.80%
  epoch  80/100 | train_acc= 36.80%
  epoch  90/100 | train_acc= 36.80%
  epoch 100/100 | train_acc= 36.80%
train=36.80%  test=33.79%

hidden_size = 50
  epoch  10/100 | train_acc= 75.20%
  epoch  20/100 | train_acc= 75.50%
  epoch  30/100 | train_acc= 75.60%
  epoch  40/100 | train_acc= 76.00%
  epoch  50/100 | train_acc= 75.90%
  epoch  60/100 | train_acc= 75.90%
  epoch  70/100 | train_acc= 76.10%
  epoch  80/100 | train_acc= 76.10%
  epoch  90/100 | train_acc= 76.40%
  epoch 100/100 | train_acc= 76.40%
train=76.40%  test=65.06%

hidden_size = 100
  epoch  10/100 | train_acc= 82.10%
  epoch  20/100 | train_acc= 83.10%
  epoch  30/100 | train_acc= 83.60%
  epoch  40/100 | train_acc= 83.90%
  epoch  50/100 | train_acc= 